In [ ]:
from NumOpt import Opti,ca 
import numpy as np 

opti=Opti()

x=opti.variable(init_guess=np.array([0.0]))
p=opti.parameter(value=0.5)
f=(x-p)**2
opti.minimize(f)
opti.ipopt_solver()
nlp:ca.Function=opti.to_function("nlp",[x,p],[x],["x0","p"],["x_star"])

C=ca.CodeGenerator("func.c")
C.add(nlp)
C.generate()

print(ca.GlobalOptions.getCasadiPath())
! gcc -O3 -fPIC -shared -o func.dll ./func.c -I D:/micromamba/envs/py12/Lib/site-packages/casadi/include/ -L D:/micromamba/envs/py12/Lib/site-packages/casadi -lipopt -lm
# ! cl /LD /O2 func.c /I D:/micromamba/envs/py12/Lib/site-packages/casadi/include /Fe:func.dll /link /LIBPATH:D:/micromamba/envs/py12/Lib/site-packages/casadi ipopt.lib

In [ ]:
import casadi as ca
import numpy as np
import subprocess

x = ca.MX.sym("x")
y = ca.MX.sym("y")

mingw_jit_options = {
    "jit": True,
    "compiler": "shell",
    "jit_options": {
        "compiler": "gcc",  # Force GCC instead of cl.exe
        "linker": "gcc",  # Force GCC for linking
        "compiler_setup": "-fPIC -c",  # GNU compilation flags
        "linker_setup": "-shared",  # GNU linking flags
        "compiler_output_flag": "-o ",
        "linker_output_flag": "-o ",
        "flags": [
            "-O3",
            "-lm",
            "-march=native",
            "-ffast-math",
            # "-flto",
            # "-fopenmp",
        ],  # Optimize output code
        "verbose": True,
    },
}

funcb = ca.Function("funcb", [x], [ca.sin(x)], ["x"], ["o1"], mingw_jit_options)

import gc
del funcb
gc.collect()

In [ ]:
from NumOpt import ca,Opti 

x=ca.MX.sym("x")
y=ca.MX.sym("x")
f=(x-0.5)**2+(y-2.0)**2
func=ca.Function("func",[x,y],[f])
func_jac=func.jacobian()
func_hess=func_jac.jacobian()

cg=ca.CodeGenerator("myfunc.c")
cg.add(func)
cg.add(func_jac)
cg.add(func_hess)
cg.generate()

! gcc -O3 -fPIC -shared -o myfunc.dll myfunc.c -lm


func=ca.external("func","myfunc.dll")
func(0.5,2.0)

opti=Opti()
x=opti.variable(init_guess=5.0,lower_bound=0.0,upper_bound=10.0)
y=opti.variable(init_guess=4.0,lower_bound=-10.0,upper_bound=5.0)
z=func(x,y)

opti.minimize(z)

opti.ipopt_solver()

sol=opti.solve()
print(sol.value(x),sol.value(y))